### Import Libraries

In [ ]:
import os
import re
import numpy as np
import pandas as pd

# Import findspark to locate the Spark installation on the system.
import findspark

# Initialise findspark so Python can find and use PySpark libraries
findspark.init() 

# Import the PySpark package to access Spark functionality
import pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import * #col, count, trim, length, lower 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
import seaborn as sns

### Startup Spark

In [ ]:
spark = (
    SparkSession.builder
    .appName("VictorianAuthorLSTM")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

### Load Dataset

In [ ]:
DATA_PATH = "C:\Datasets\Gungor_2018_VictorianAuthorAttribution_data-train.csv"

df_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "ISO-8859-1")
    .csv(DATA_PATH)
)

### Show the number of rows and schema

In [ ]:
print("Number of rows:", df_spark.count())

df_spark.printSchema()

### Show a few number of records

In [ ]:
df_spark.select("text", "author").show(10)

### Group and Count the number of authors

In [ ]:
author_counts = (
    df_spark
    .groupBy("author")
    .count()
    .orderBy("author")
)

author_counts.show(50)

### Graphical Presentation of Authors

In [ ]:
author_counts_pd = author_counts.toPandas()

plt.figure(figsize=(14, 6))

plt.bar(
    author_counts_pd["author"].astype(str),
    author_counts_pd["count"]
)

plt.xlabel("Author ID")
plt.ylabel("Number of Text Fragments")
plt.title("Distribution of Training Text Fragments by Author")
plt.xticks(rotation=90)

plt.show()

In [ ]:
df_spark.select(
    "author",
    "text"
).show(
    5,
    truncate=200
)

### Clean text using native PySpark

In [ ]:
#from pyspark.sql.functions import lower
df_clean = (
    df_spark
    .withColumn(
        "clean_text",
        lower(col("text"))
    )
    .withColumn(
        "clean_text",
        regexp_replace(
            col("clean_text"),
            r"[^a-z\s]",
            " "
        )
    )
    .withColumn(
        "clean_text",
        regexp_replace(
            col("clean_text"),
            r"\s+",
            " "
        )
    )
    .withColumn(
        "clean_text",
        trim(col("clean_text"))
    )
)

### Show Character Count per Row

In [ ]:
df_spark.select(
    "author",
    length("text").alias("char_count")
    ).orderBy(
    "char_count",
    descending=False
    ).show(10, truncate=200)

### Randomly select a Maximum number of Samples (1000) per author and delete the null rows

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, rand, col

SAMPLES_PER_AUTHOR = 800

window_by_author = Window.partitionBy("author").orderBy(rand(seed=42))

df_sample = (
    df_clean
    .filter(col("author").isNotNull() & col("clean_text").isNotNull())
    .withColumn("row_num", row_number().over(window_by_author))
    .filter(col("row_num") <= SAMPLES_PER_AUTHOR)
    .drop("row_num")
)

print("Balanced sample size:", df_sample.count())
print("Number of authors:", df_sample.select("author").distinct().count())

df_sample.groupBy("author").count().orderBy("author").show(50)

### Convert Spark data to Pandas
  *The reason for converting to Pandas is that TensorFlow/Keras will be used to train the LSTM*

In [ ]:
pdf = (
    df_sample
    .select("author", "clean_text")
    .toPandas()
)

pdf = pdf.dropna(subset=["author", "clean_text"]).reset_index(drop=True)

print("Pandas shape:", pdf.shape)
print("Authors represented:", pdf["author"].nunique())

### Check for duplicates

In [ ]:
print(pdf.duplicated())

### Split the Data into Training and Test Data

In [ ]:
X_text_train, X_text_test, y_train_raw, y_test_raw = train_test_split(
    pdf["clean_text"].values,
    pdf["author"].values,
    test_size=0.20,
    random_state=42,
    stratify=pdf["author"].values   #ensures that the train and test sets have approximately the same distribution of authors as the original dataset
)

print("Training texts:", len(X_text_train))
print("Testing texts:", len(X_text_test))

### Encode the authors

In [ ]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)

NUM_CLASSES = len(label_encoder.classes_)

print("Number of authors:", NUM_CLASSES)
print("Number of classes:", label_encoder.classes_)

### Graphical Presentation of Authors after the limit was set to 1000 characters

In [ ]:
#Author Destribution
author_distribution = pdf["author"].value_counts().sort_index()

plt.figure(figsize=(14, 6))

author_distribution.plot(kind="bar")
plt.xlabel("Author")
plt.ylabel("Number of Text Samples")
plt.title("Distribution of Victorian Author Text Samples")
plt.tight_layout()

plt.show()

### Tokenise

In [ ]:
MAX_WORDS = 30000
MAX_SEQUENCE_LENGTH = 500    # Set to this number to reduce CPU and Memory issues when model is created

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<TTT>",
    filters=""
)

tokenizer.fit_on_texts(X_text_train)

X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_text_train),
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    tokenizer.texts_to_sequences(X_text_test),
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Vocabulary learned from training data:", len(tokenizer.word_index))

### Bidirectional LSTM Architecture

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, SpatialDropout1D
from tensorflow.keras.models import Sequential

model = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,)),

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=160,
        mask_zero=True
    ),

    SpatialDropout1D(0.20),

    Bidirectional(
        LSTM(
            160,
            return_sequences=False,
            dropout=0.25
        )
    ),

    Dense(128, activation="relu"),
    Dropout(0.40),

    Dense(NUM_CLASSES, activation="softmax")
])

model.summary()

### Build Long Short-Term Memory Recurrent Neural Network

In [ ]:
NUM_CLASSES = len(
    label_encoder.classes_
)

model = Sequential([

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
    ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=False,
            dropout=0.2
        )
    ),

    Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

## Compile the Model
*The learning rate is set to 0.0005 and gradient clipping is enabled. This is intended to make training more stable on the larger model.*

In [ ]:
optimizer = Adam(
    learning_rate=0.0005,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Train the Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=15,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

### Accuracy Plot

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "LSTM Training and Validation Accuracy"
)

plt.legend()

plt.tight_layout()

plt.show()

### Plot Loss

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "LSTM Training and Validation Loss"
)

plt.legend()

plt.tight_layout()

plt.show()

### Evaluate LSTM model

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(
    f"Test Loss: {test_loss:.4f}"
)

print(
    f"Test Accuracy: {test_accuracy:.4f}"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

### LSTM Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(
    f"Test Loss: {test_loss:.4f}"
)

print(
    f"Test Accuracy: {test_accuracy:.4f}"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

### Generate Prediction for Re-Trained model

In [ ]:
probabilities = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    probabilities,
    axis=1
)

### Precision, recall and F1-score

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            str(x)
            for x in label_encoder.classes_
        ],
        zero_division=0
    )
)

# ReTraining

### Retraining LSTM Architecture

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, SpatialDropout1D
from tensorflow.keras.models import Sequential

model_retr = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,)),

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=160,
        mask_zero=True
    ),

    SpatialDropout1D(0.20),

    Bidirectional(
        LSTM(
            160,
            return_sequences=False,
            dropout=0.25
        )
    ),

    Dense(128, activation="relu"),
    Dropout(0.40),

    Dense(NUM_CLASSES, activation="softmax")
])


### Recompile Model

In [ ]:
optimizer = Adam(
    learning_rate=0.0005,
    clipnorm=1.0
)

model_retr.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Actual Model Retraining

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history_retr = model_retr.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

### Retrained Model Accuracy Plot

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history_retr.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_retr.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Retrained LSTM Training and Validation Accuracy"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history_retr.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_retr.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "LSTM Training and Validation Loss"
)

plt.legend()

plt.tight_layout()

plt.show()

### Re-trained LSTM Evaluation

In [ ]:
test_loss, test_accuracy = model_retr.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(
    f"Test Loss: {test_loss:.4f}"
)

print(
    f"Test Accuracy: {test_accuracy:.4f}"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

### Generate Prediction for Re-Trained model

In [ ]:
probabilities = model_retr.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    probabilities,
    axis=1
)

### Precision, recall and F1-score

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            str(x)
            for x in label_encoder.classes_
        ],
        zero_division=0
    )
)

# Create the Bot

In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

### Create a function to predict an Author

In [ ]:
def predict_author(text):
    
    # Clean input
    cleaned = clean_text(text)
    
    # Convert words to numerical sequence
    sequence = tokenizer.texts_to_sequences(
        [cleaned]
    )
    
    # Pad sequence
    padded = pad_sequences(
        sequence,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post"
    )
    
    # Predict
    probabilities = model_retr.predict(
        padded,
        verbose=0
    )[0]
    
    # Get best prediction
    predicted_index = np.argmax(
        probabilities
    )
    
    predicted_author = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    confidence = probabilities[predicted_index]
    
    return predicted_author, confidence

### Bot Input Prompt

In [ ]:
print("Enter text (press Enter twice to finish):")

lines = []

while True:
    line = input()
    if line == "":
        break
    lines.append(line)

text = "\n".join(lines)

author, confidence = predict_author(text)
print(f"\nPredicted Author: {author}")

print(f"Confidence: {confidence * 100:.2f}%")